# Harbour watch: the capstone

Everything in one scene: sensors colored and sized from their data, vessel
tracks that march segment by segment, dwell zones that appear and lapse, labels,
folders, the legend, a selection moment, and the whole thing shipped as one
file. Each piece has its own notebook; this is what they look like together.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(42)

# Sensors: static instruments, colored by reading, sized by volume.
n = 300
sensors = pd.DataFrame({
    "lat": 36.02 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.09, n),
    "site": [f"S{i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "volume": rng.integers(10, 500, n),
    "status": rng.choice(["Active", "Idle"], n, p=[0.8, 0.2]),
})

# Vessels: three tracks, one timestamp per vertex.
steps = 48
tracks = []
for vessel, (lat0, lon0) in [("Vessel A", (36.00, -5.85)),
                             ("Vessel B", (35.94, -5.78)),
                             ("Vessel C", (36.08, -5.90))]:
    tracks.append(pd.DataFrame({
        "vessel": vessel,
        "lat": lat0 + np.cumsum(rng.normal(0.002, 0.003, steps)),
        "lon": lon0 + np.cumsum(rng.normal(0.009, 0.004, steps)),
        "timestamp": pd.date_range("2026-08-01 00:00", periods=steps,
                                   freq="30min", tz="UTC"),
    }))
tracks = pd.concat(tracks, ignore_index=True)

# Dwells: zones that exist for a while, with a risk rating.
def rect(name, clat, clon, start, end, risk):
    h, w = 0.014, 0.02
    return [{"dwell": name, "vertex": v, "lat": clat + dy, "lon": clon + dx,
             "times": [start, end], "risk": risk}
            for v, (dy, dx) in enumerate([(-h, -w), (-h, w), (h, w), (h, -w)])]

dwells = pd.DataFrame(
    rect("Dwell North", 36.13, -5.52, "2026-08-01 02:00", "2026-08-01 14:00", "high")
    + rect("Dwell South", 35.93, -5.42, "2026-08-01 06:00", "2026-08-01 22:00", "medium")
    + rect("Dwell East", 36.05, -5.28, "2026-08-01 10:00", "2026-08-02 00:00", "low")
)

## One map, a handful of calls

Sensors, vessels, dwells — then the furniture: the legend, a nautical scale bar,
and a draw toolbar for sketching an AOI right on the scene.

In [ ]:
m = Map(height="620px")
m.add_circle_markers(sensors, name="Sensors",
                     layer_group=["Harbour", "status"],
                     color_col="reading", radius_col="volume",
                     radius_range=(3, 12),
                     popup_fields=["site", "reading", "volume"],
                     popup_names=["Site", "Reading", "Volume"])
m.add_line(tracks, line_id_col="vessel", order_col="timestamp",
           name="vessel", layer_group="Vessels", weight=3,
           color_col="vessel")
m.add_polygon(dwells, shape_id_col="dwell", order_col="vertex",
              name="dwell", label="dwell", layer_group="Dwells",
              color="crimson", fill_opacity=0.25,
              popup_fields=["risk"], popup_names=["Risk"])
m.configure_group("Harbour", collapsed=False)
m.configure_legend(show=True, title="Harbour Watch")
m.show_scale = True
m.configure_scale(units="nautical", position="bottom-right")
m.configure_draw(show=True, tools=["rectangle", "polygon"])
m

## Set it in motion

One slider serves everything: tracks march segment by segment with a trailing
window and fade, dwell zones appear and lapse over their own intervals (their
labels with them), and the sensors — timeless — stay put, which is itself the
signal that they are instruments, not events.

In [ ]:
m.make_time_layer(group="Vessels", duration="PT3H", fade=True)
m.make_time_layer(group="Dwells")
m.configure_time(period="PT1H", speed=2, loop=True, position="bottom-center");

## The investigation moment

A report comes in about the northern dwell: select it, zoom to it, flag the
vessel of interest — then put the room back the way it was.

In [ ]:
m.select("Dwell North", scope="Dwells", zoom=True, zoom_offset=-1)
m.highlight("Vessel A", color="#ffcc00", lines={"weight": 6});

In [ ]:
m.select(None, scope="Dwells")
m.highlight(None)
m.fit_bounds(m.bounds_of());

## Ship it

One file, no backend: sidebar, legend, labels, and the time slider all work in
the export.

In [ ]:
from pathlib import Path
m.save("showcase.html")
print(f"showcase.html: {Path('showcase.html').stat().st_size / 1024:,.0f} KB")

Every piece here has a notebook of its own: data formats (**02**), styling
(**03**), folders (**04**), targeting (**05**), time (**06**), popups and labels
(**07**), export (**08**), the legend (**11**) — and the same scene runs live
against Shiny in `shiny/`.